In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

%matplotlib inline

## Scikit Pipeline Demo - organizing data processing workflow

In [2]:
X, y = make_classification(n_samples=100, n_features=20, n_classes=2)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, train_size=0.75)

In [4]:
pipeline = Pipeline([
    ('scaler', StandardScaler(with_mean=True, with_std=True)),
    ('svc', SVC(C=1.0, kernel='rbf'))
])

pipeline = pipeline.fit(X_train, y_train)

score_train = pipeline.score(X_train, y_train)
score_test = pipeline.score(X_test, y_test)

print(score_train)
print(score_test)

0.9866666666666667
0.96


In [5]:
pipeline = Pipeline([
    ('scaler', StandardScaler(with_mean=True, with_std=True)),
    ('svc', SVC())
])

pipeline = pipeline.set_params(svc__C=0.5)
pipeline = pipeline.set_params(svc__kernel='linear')

pipeline = pipeline.fit(X_train, y_train)

score_train = pipeline.score(X_train, y_train)
score_test = pipeline.score(X_test, y_test)

print(score_train)
print(score_test)

0.9333333333333333
1.0


## Scikit GridSearch Demo - optimizing hyperparameters with a simple grid search

In [6]:
gs_parameters = {
    'svc__C': [0.25, 0.50, 0.75, 1, 5, 10, 15, 20, 25],
    'svc__kernel': ('linear', 'rbf')
}

gs = GridSearchCV(pipeline, gs_parameters)

gs.fit(X_train, y_train)

GridSearchCV(estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svc', SVC(C=0.5, kernel='linear'))]),
             param_grid={'svc__C': [0.25, 0.5, 0.75, 1, 5, 10, 15, 20, 25],
                         'svc__kernel': ('linear', 'rbf')})

In [7]:
gs.best_params_

{'svc__C': 1, 'svc__kernel': 'rbf'}

In [8]:
gs.best_estimator_

Pipeline(steps=[('scaler', StandardScaler()), ('svc', SVC(C=1))])

In [9]:
best_model = gs.best_estimator_

best_model = best_model.fit(X_train, y_train)

score_train = best_model.score(X_train, y_train)
score_test = best_model.score(X_test, y_test)

print(score_train)
print(score_test)

0.9866666666666667
0.96


In [10]:
gs.cv_results_

{'mean_fit_time': array([0.00167928, 0.00356956, 0.00250149, 0.00112462, 0.00076985,
        0.00067739, 0.00076632, 0.00072207, 0.00092688, 0.00066042,
        0.00094523, 0.00064416, 0.00089879, 0.00063024, 0.00089068,
        0.00062704, 0.00089431, 0.00063281]),
 'std_fit_time': array([5.23456938e-04, 2.20699433e-03, 7.62342473e-04, 5.01462981e-04,
        1.44979278e-04, 5.20933193e-05, 1.49335578e-04, 5.68836152e-05,
        3.82778058e-04, 2.87797989e-05, 3.75939383e-04, 1.00174397e-05,
        3.73812153e-04, 2.28981155e-06, 3.76715280e-04, 2.41262639e-06,
        3.80230769e-04, 6.18676689e-06]),
 'mean_score_time': array([0.00110927, 0.00226455, 0.00106649, 0.00062838, 0.00033784,
        0.00031934, 0.00028782, 0.00033474, 0.00027328, 0.00031614,
        0.00027299, 0.00029745, 0.00026217, 0.00029364, 0.00026135,
        0.00029373, 0.00026469, 0.00029607]),
 'std_score_time': array([6.21767746e-04, 2.11000680e-03, 6.07983110e-04, 2.55106213e-04,
        6.95703616e-05, 9.92

## Optuna Demo - optimizing hyperparameters with an optimization framework

In [11]:
# pip install optuna
# conda install conda-forge::optuna

import optuna

In [12]:
def objective_function(trial):
    x = trial.suggest_float('x', -np.pi, np.pi)
    return (np.sin(x) - 0.25 * np.pi) ** 2

study = optuna.create_study()

study.optimize(objective_function, n_trials=100)

study.best_params

[I 2025-02-24 11:37:28,309] A new study created in memory with name: no-name-5e6bd2d2-e8a2-47e1-a5ae-83f0c47066e6
[I 2025-02-24 11:37:28,311] Trial 0 finished with value: 0.02811261877697675 and parameters: {'x': 1.878385173722843}. Best is trial 0 with value: 0.02811261877697675.
[I 2025-02-24 11:37:28,312] Trial 1 finished with value: 1.746531193545638 and parameters: {'x': -2.575704467955842}. Best is trial 0 with value: 0.02811261877697675.
[I 2025-02-24 11:37:28,312] Trial 2 finished with value: 0.04425691690817308 and parameters: {'x': 1.4788013252611476}. Best is trial 0 with value: 0.02811261877697675.
[I 2025-02-24 11:37:28,312] Trial 3 finished with value: 3.1750579943739825 and parameters: {'x': -1.4867606291306603}. Best is trial 0 with value: 0.02811261877697675.
[I 2025-02-24 11:37:28,313] Trial 4 finished with value: 0.20938148608398138 and parameters: {'x': 0.333990844275319}. Best is trial 0 with value: 0.02811261877697675.
[I 2025-02-24 11:37:28,313] Trial 5 finished 

{'x': 2.235868463176809}

In [13]:
def objective_function(trial):
    svc_C = trial.suggest_float('svc__C', 0, 25)
    svc_kernel = trial.suggest_categorical('svc__kernel', ['linear', 'rbf'])

    pipeline = Pipeline([
        ('scaler', StandardScaler(with_mean=True, with_std=True)),
        ('svc', SVC())
    ])

    pipeline = pipeline.set_params(svc__C=svc_C)
    pipeline = pipeline.set_params(svc__kernel=svc_kernel)

    pipeline = pipeline.fit(X_train, y_train)

    score_test = pipeline.score(X_test, y_test)

    return score_test

study = optuna.create_study(direction='maximize')

study.optimize(objective_function, n_trials=100)

study.best_params

[I 2025-02-24 11:37:28,591] A new study created in memory with name: no-name-c16cbf23-c273-4310-9d19-9e502a83a66a
[I 2025-02-24 11:37:28,595] Trial 0 finished with value: 0.92 and parameters: {'svc__C': 17.55434639011894, 'svc__kernel': 'rbf'}. Best is trial 0 with value: 0.92.
[I 2025-02-24 11:37:28,598] Trial 1 finished with value: 0.92 and parameters: {'svc__C': 24.996059634882972, 'svc__kernel': 'rbf'}. Best is trial 0 with value: 0.92.
[I 2025-02-24 11:37:28,600] Trial 2 finished with value: 0.84 and parameters: {'svc__C': 15.556950834247715, 'svc__kernel': 'linear'}. Best is trial 0 with value: 0.92.
[I 2025-02-24 11:37:28,602] Trial 3 finished with value: 1.0 and parameters: {'svc__C': 0.5115552531168022, 'svc__kernel': 'linear'}. Best is trial 3 with value: 1.0.
[I 2025-02-24 11:37:28,604] Trial 4 finished with value: 0.92 and parameters: {'svc__C': 7.031667223823473, 'svc__kernel': 'rbf'}. Best is trial 3 with value: 1.0.
[I 2025-02-24 11:37:28,606] Trial 5 finished with value

{'svc__C': 0.5115552531168022, 'svc__kernel': 'linear'}

In [14]:
# svc_C = study.best_params['svc__C']
# svc_kernel = study.best_params['svc__kernel']

# best_model = Pipeline([
#         ('scaler', StandardScaler(with_mean=True, with_std=True)),
#         ('svc', SVC(C=svc_C, kernel=svc_kernel))
#     ])

best_model = Pipeline([
        ('scaler', StandardScaler(with_mean=True, with_std=True)),
        ('svc', SVC())
    ])

best_model.set_params(**study.best_params)

best_model = best_model.fit(X_train, y_train)

score_train = best_model.score(X_train, y_train)
score_test = best_model.score(X_test, y_test)

print(score_train)
print(score_test)

0.9466666666666667
1.0
